# Data Preprocessing for Sentiment Analysis

This notebook handles data cleaning, preprocessing, and labeling for the sentiment analysis project.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data if needed
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('vader_lexicon')

In [ ]:
# Load raw data
raw_data = pd.read_csv('../data/raw/reviews_raw.csv')
print(f"Loaded {len(raw_data)} raw reviews")
raw_data.head()

In [ ]:
# Data exploration
print("Dataset Info:")
print(raw_data.info())
print("\nMissing values:")
print(raw_data.isnull().sum())
print("\nRating distribution:")
print(raw_data['rating'].value_counts().sort_index())

In [ ]:
# Visualize rating distribution
plt.figure(figsize=(8, 6))
sns.countplot(data=raw_data, x='rating')
plt.title('Distribution of Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

In [ ]:
# Text preprocessing functions
def clean_text(text):
    """Clean and preprocess text data."""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def preprocess_text(text):
    """Advanced text preprocessing with tokenization and lemmatization."""
    if not text:
        return ""
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return ' '.join(tokens)

In [ ]:
# Apply text cleaning
raw_data['review_text_clean'] = raw_data['review_text'].apply(clean_text)
raw_data['review_text_processed'] = raw_data['review_text_clean'].apply(preprocess_text)

print("Text cleaning completed!")
print("\nExample of original vs cleaned text:")
for i in range(3):
    print(f"Original: {raw_data.iloc[i]['review_text']}")
    print(f"Cleaned: {raw_data.iloc[i]['review_text_clean']}")
    print(f"Processed: {raw_data.iloc[i]['review_text_processed']}")
    print("-" * 50)

In [ ]:
# Create sentiment labels based on ratings
def create_sentiment_label(rating):
    """Create sentiment labels from ratings."""
    if rating >= 4:
        return 'positive'
    elif rating <= 2:
        return 'negative'
    else:
        return 'neutral'

# Apply sentiment labeling
raw_data['sentiment_label'] = raw_data['rating'].apply(create_sentiment_label)

print("Sentiment label distribution:")
print(raw_data['sentiment_label'].value_counts())

In [ ]:
# Add confidence scores using TextBlob
def get_confidence_score(text):
    """Get confidence score for sentiment using TextBlob."""
    if not text:
        return 0.5
    
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    
    # Convert polarity to confidence score
    confidence = abs(polarity)
    return round(confidence, 2)

# Apply confidence scoring
raw_data['confidence_score'] = raw_data['review_text_processed'].apply(get_confidence_score)

print("Confidence score statistics:")
print(raw_data['confidence_score'].describe())

In [ ]:
# Create final processed dataset
processed_data = raw_data[[
    'review_id', 'review_text_processed', 'sentiment_label', 'confidence_score'
]].copy()

# Rename column for consistency
processed_data = processed_data.rename(columns={
    'review_text_processed': 'review_text_clean'
})

print("Final processed dataset:")
print(processed_data.head())
print(f"\nTotal processed reviews: {len(processed_data)}")

In [ ]:
# Save processed data
processed_data.to_csv('../data/processed/reviews_clean.csv', index=False)
print("Processed data saved to ../data/processed/reviews_clean.csv")

In [ ]:
# Final visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Sentiment distribution
sns.countplot(data=processed_data, x='sentiment_label', ax=axes[0])
axes[0].set_title('Sentiment Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

# Confidence score distribution
sns.histplot(data=processed_data, x='confidence_score', bins=20, ax=axes[1])
axes[1].set_title('Confidence Score Distribution')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()